# Temporal GNN training

Arsitektur mempertahankan graph pada seluruh 101 frame: dua GCN memproduksi
embedding setiap frame, mean pooling menggabungkan node valid, dan GRU memodelkan
urutan waktu sebelum klasifikasi biner. Split pertandingan sama dengan baseline.

Eksekusi saat ini adalah **smoke test**. Mode final ditolak otomatis sampai
seluruh anotasi dan minimum kelas per split memenuhi quality gate.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.graph_cache import audit_graph_cache, build_graph_cache
from src.research_readiness import build_model_manifest
from src.temporal_gnn_training import prepare_gnn_dataset, train_temporal_gnn

DATA_DIR = PROJECT_ROOT / 'data' / 'sportec_idsse'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
GRAPH_CACHE_DIR = PROCESSED_DIR / 'graph_cache'
GRAPH_CACHE_MANIFEST_PATH = PROCESSED_DIR / 'graph_cache_manifest.csv'
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts' / 'temporal_gnn_smoke'
FINAL_RUN = False

In [2]:
annotations = pd.read_csv(PROCESSED_DIR / 'gegenpressing_annotation_manifest.csv')
cache_manifest = build_graph_cache(
    annotations,
    DATA_DIR,
    GRAPH_CACHE_DIR,
    overwrite=False,
)
cache_manifest.to_csv(GRAPH_CACHE_MANIFEST_PATH, index=False)

cache_checks = audit_graph_cache(cache_manifest, annotations)
display(cache_checks.to_frame())
assert cache_checks.all(), 'Graph cache audit gagal.'
print('Cached graph sequences:', len(cache_manifest))

,passed
candidate_id_unique,True
candidate_ids_complete,True
all_101_frames,True
all_23_nodes,True
all_6_node_features,True
all_92_edges_per_frame,True
all_nodes_valid,True


Cached graph sequences: 264


In [3]:
episodes = pd.read_csv(PROCESSED_DIR / 'episode_manifest.csv')
model_manifest = build_model_manifest(episodes, annotations)
dataset = prepare_gnn_dataset(model_manifest, cache_manifest)

metrics, predictions, history, readiness = train_temporal_gnn(
    dataset,
    annotations,
    GRAPH_CACHE_DIR,
    ARTIFACT_DIR,
    final_run=FINAL_RUN,
    random_state=42,
    batch_size=4,
    max_epochs=25 if not FINAL_RUN else 100,
    patience=6 if not FINAL_RUN else 15,
)

print('Final-training readiness:')
display(readiness.to_frame())
display(metrics[[
    'model', 'split', 'samples', 'positive_rate', 'threshold',
    'precision', 'recall', 'f1', 'roc_auc', 'pr_auc', 'loss',
]])
print('Best validation loss:', history['validation_loss'].min())
print('Artifacts:', ARTIFACT_DIR)
if not FINAL_RUN:
    print('PERINGATAN: metrik pilot tidak boleh dilaporkan sebagai hasil final.')

Final-training readiness:


,passed
all_annotations_completed,False
all_labels_valid,True
candidate_id_unique,True
confidence_complete,True
confidence_valid,True
uncertain_have_notes,True
annotator_complete,True
both_classes_in_every_split,True
minimum_per_class_in_every_split,False
uncertain_excluded,True


,model,split,samples,positive_rate,threshold,precision,recall,f1,roc_auc,pr_auc,loss
0,temporal_gnn,train,20,0.850000,0.750985,1.0,0.588235,0.740741,1.0,1.0,0.324785
1,temporal_gnn,validation,3,0.666667,0.750985,1.0,1.000000,1.000000,1.0,1.0,0.611859
2,temporal_gnn,test,3,0.666667,0.750985,0.0,0.000000,0.000000,1.0,1.0,0.597790


Best validation loss: 0.6118592619895935
Artifacts: C:\VAEP-Track\artifacts\temporal_gnn_smoke
PERINGATAN: metrik pilot tidak boleh dilaporkan sebagai hasil final.
